In [2]:
# 파일명: run_beit3_inference.py

import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import os
from pathlib import Path
import json
from collections import Counter

# BEiT-3 모델 관련 import
from modeling_finetune import beit3_large_patch16_768_vqav2
from transformers import XLMRobertaTokenizer

# --- 1. 설정 (Configuration) ---
print("--- [단계 1/2] BEiT-3 후보 답변 생성 시작 ---")
print("1. 설정을 초기화합니다...")

# 경로 설정
BEIT3_MODEL_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
SPM_MODEL_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
VQA_ANNOTATION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"
DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
BASE_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
# 중간 결과를 저장할 파일
INTERMEDIATE_CSV_PATH = "beit3_candidates.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {DEVICE}")

# --- 2. BEiT-3 모델 로드 ---
print("\n2. BEiT-3 모델을 로드합니다...")
beit3_tokenizer = XLMRobertaTokenizer(SPM_MODEL_PATH)
beit3_model = beit3_large_patch16_768_vqav2()
checkpoint = torch.load(BEIT3_MODEL_PATH, map_location='cpu')
beit3_model.load_state_dict(checkpoint['model'])
beit3_model.to(DEVICE)
beit3_model.eval()
print("BEiT-3 모델 로드 완료.")

# --- 3. 답변 사전 및 이미지 전처리기 준비 ---
print("\n3. 답변 사전과 이미지 전처리기를 준비합니다...")
try:
    with open(VQA_ANNOTATION_PATH, 'r') as f:
        vqa_data = json.load(f)
    raw_annotations = vqa_data['annotations']
    answer_counter = Counter(ann['multiple_choice_answer'] for ann in raw_annotations)
    top_answers = answer_counter.most_common(3129)
    beit3_answer_map = {str(i): answer for i, answer in enumerate(top_answers)}
    print("BEiT-3 답변 사전 생성 완료.")
except Exception as e:
    print(f"오류: BEiT-3 답변 사전 준비 중 문제가 발생했습니다: {e}")
    exit()

beit3_transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# --- 4. 데이터 로드 및 BEiT-3 추론 루프 ---
print(f"\n4. {DEV_TEST_CSV_PATH} 파일을 읽어 추론을 시작합니다...")
df_test = pd.read_csv(DEV_TEST_CSV_PATH)
candidate_results = []

for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0], desc="BEiT-3 추론 진행률"):
    test_id = row['ID']
    relative_img_path = row['img_path']
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]
    
    image_filename = os.path.basename(relative_img_path)
    full_image_path = Path(BASE_IMAGE_DIR) / image_filename

    try:
        image = Image.open(full_image_path).convert("RGB")
        image_tensor_beit3 = beit3_transform(image).unsqueeze(0).to(DEVICE)
        encoding_beit3 = beit3_tokenizer(
            question, padding='max_length', truncation=True,
            max_length=40, return_tensors='pt'
        )
        question_tokens = encoding_beit3['input_ids'].to(DEVICE)
        padding_mask = encoding_beit3['attention_mask'].to(DEVICE)

        with torch.no_grad():
            output_logits_beit3 = beit3_model(
                image=image_tensor_beit3, 
                question=question_tokens, 
                padding_mask=padding_mask
            )
        
        predicted_index_beit3 = output_logits_beit3.argmax(-1).item()
        candidate_answer = beit3_answer_map[str(predicted_index_beit3)]
        
        # 다음 단계를 위해 필요한 모든 정보를 저장
        candidate_results.append({
            'ID': test_id,
            'Question': question,
            'A': choices[0],
            'B': choices[1],
            'C': choices[2],
            'D': choices[3],
            'candidate_answer': candidate_answer # BEiT-3가 생성한 후보 답변
        })

    except Exception as e:
        print(f"\n오류: ID '{test_id}' 처리 중 예외 발생: {e}")
        candidate_results.append({
            'ID': test_id, 'Question': question,
            'A': choices[0], 'B': choices[1], 'C': choices[2], 'D': choices[3],
            'candidate_answer': 'Error'
        })

# --- 5. 중간 결과 저장 ---
df_candidates = pd.DataFrame(candidate_results)
df_candidates.to_csv(INTERMEDIATE_CSV_PATH, index=False)

print("\n" + "="*50)
print(f"BEiT-3 후보 답변 생성이 완료되었습니다.")
print(f"중간 결과가 {INTERMEDIATE_CSV_PATH} 파일에 저장되었습니다.")
print("다음 단계로 이 파일을 사용하세요.")
print("="*50)

--- [단계 1/2] BEiT-3 후보 답변 생성 시작 ---
1. 설정을 초기화합니다...
사용 디바이스: cuda

2. BEiT-3 모델을 로드합니다...
BEiT-3 모델 로드 완료.

3. 답변 사전과 이미지 전처리기를 준비합니다...
BEiT-3 답변 사전 생성 완료.

4. /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv 파일을 읽어 추론을 시작합니다...


BEiT-3 추론 진행률: 100%|██████████| 60/60 [00:10<00:00,  5.84it/s]



BEiT-3 후보 답변 생성이 완료되었습니다.
중간 결과가 beit3_candidates.csv 파일에 저장되었습니다.
다음 단계로 이 파일을 사용하세요.
